# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yousefwerida28/Flyrank-ML-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

In [25]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Unit of analysis + time window

Unit of analysis: The intended grain of fact_content_daily_performance is one content page for one client on one report date. I found 6,390 exact duplicate page-client-date records, so these duplicates should be removed before analysis.

Time window: I will use March 1–31, 2026 as the development window. The fact_content_daily_performance table contains daily performance observations identified by report_date. The final month, June 2026, will be kept as a sealed test window.


In [26]:
con.sql("""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date
0,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

Selected features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, and ga4_total_engagement_sec. These features represent different dimensions of content performance: search visibility, search clicks, search ranking, traffic, and engagement. They will be scaled before clustering so that features with larger numeric ranges do not dominate the clustering.

Label/proxy: None initially — clustering is unsupervised and discovers the groups from the features.

Contexts:client_hash_id , content_hash_id , report_date.

Exclude:ai_chatgpt → ai_other (Specific AI traffic sources are not needed for our general performance clustering.)



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [27]:
df = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 10
""").df()

display(df)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01
5,2025-01-27,client_9958f0a7ae1df715,content_c782fa8abd4fce5e,True,True,True,False,21,0,1050,...,0,0,0,0,0,0,0,0,0,2025-01
6,2025-01-27,client_9958f0a7ae1df715,content_ae5e5fd6edff550f,True,True,True,False,13,0,127,...,0,0,0,0,0,0,0,0,0,2025-01
7,2025-01-27,client_9958f0a7ae1df715,content_a64143f6e4a21ffe,True,True,True,False,29,0,356,...,0,0,0,0,0,0,0,0,0,2025-01
8,2025-01-27,client_9958f0a7ae1df715,content_e281674658070602,True,True,True,False,5,0,103,...,0,0,0,0,0,0,0,0,0,2025-01
9,2025-01-27,client_9958f0a7ae1df715,content_658f53fa439c66ca,True,True,True,False,8,0,304,...,0,0,0,0,0,0,0,0,0,2025-01


**Grain verfication **


In [28]:
grain_check = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count
0,client_06d356715a8ff3b6,content_03a8b5950a52518a,2026-06-13,2
1,client_1a730cb2640a1abf,content_6604767cde89152e,2026-06-13,2
2,client_1a730cb2640a1abf,content_b5aec9a8a2ee7fb0,2026-06-13,2
3,client_1a8bf67cad4ee525,content_2630830d5f397c6c,2026-06-15,2
4,client_810019792c9b8efc,content_26b4ce630106d689,2026-06-16,2
5,client_06d356715a8ff3b6,content_f5a0c77c1826b467,2026-06-16,2
6,client_1a730cb2640a1abf,content_ac05941d77071556,2026-06-18,2
7,client_b77d0d5f08f05e64,content_6b6a532cb09428b2,2026-06-20,2
8,client_b77d0d5f08f05e64,content_52a40d3e61e4e4cc,2026-06-20,2
9,client_8ddc46da5414ffd8,content_63bd133bf8970bcd,2026-06-23,2


In [29]:
con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE client_hash_id = 'client_06d356715a8ff3b6'
      AND content_hash_id = 'content_03a8b5950a52518a'
      AND report_date = '2026-06-13'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-13,client_06d356715a8ff3b6,content_03a8b5950a52518a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-13,client_06d356715a8ff3b6,content_03a8b5950a52518a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [30]:
con.sql("""
    SELECT
        COUNT(*) AS duplicate_groups,
        SUM(row_count - 1) AS extra_duplicate_rows
    FROM (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            COUNT(*) AS row_count
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
        )
        GROUP BY
            client_hash_id,
            content_hash_id,
            report_date
        HAVING COUNT(*) > 1
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_groups,extra_duplicate_rows
0,6390,6390.0


**Count**
The March 2026 slice contains 9,841,378 rows. The available observations cover March 1, 2026 through March 31, 2026. Since the intended grain is page-day, these rows represent page-day observations.

In [31]:
march_count = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

march_count

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


**Availability**

In [32]:
availability = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS rows_with_both
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_both
0,9841378,364347


**Available When?**
### Five Features — Available When?

**1. `gsc_impressions`**
Measures how many times the page appeared in Google Search.
**Available when:** After the day's Search Console data has been recorded. It describes the page's observed search visibility.

**2. `gsc_clicks`**
Measures how many clicks the page received from Google Search.
**Available when:** After the day's Search Console data has been recorded. It describes the page's observed search clicks.

**3. `gsc_avg_position`**
Measures the page's average position in Google Search results.
**Available when:** After the day's Search Console data has been recorded. It describes the page's observed search ranking.

**4. `ga4_sessions`**
Measures the number of sessions on the page.
**Available when:** After the day's Google Analytics data has been recorded. It describes the page's observed traffic.

**5. `ga4_total_engagement_sec`**
Measures the total time users spent engaged with the page.
**Available when:** After the day's Google Analytics data has been recorded. It describes the page's observed engagement.


**Framing the five features**

In [33]:
feature_df = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_total_engagement_sec
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""").df()

feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_total_engagement_sec
0,client_65de48885f4ef01b,content_5c80451459c29b4a,2026-03-01,5,0,5.400000,1,0
1,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2026-03-01,39,0,5.666667,2,0
2,client_65de48885f4ef01b,content_e25ea7297a1dffd3,2026-03-01,179,0,5.156425,2,0
3,client_65de48885f4ef01b,content_6b0149a80607dac3,2026-03-01,72,0,7.694444,1,0
4,client_65de48885f4ef01b,content_62673eea26c31c17,2026-03-01,3282,1,6.167885,1,0


In [34]:
feature_df.shape

(364347, 8)

In [35]:
feature_df.isna().sum()

,0
client_hash_id,0
content_hash_id,0
report_date,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_sessions,0
ga4_total_engagement_sec,0


**Missing values**: The five selected features contain no missing values in the March 2026 rows where both GSC and GA4 data are available. Therefore, no missing-value treatment is required for this feature frame.

**DATA leakage experiment**

In [36]:
# Find the median number of clicks
median_clicks = feature_df["gsc_clicks"].median()

# Create a proxy label
feature_df["high_performance"] = (
    feature_df["gsc_clicks"] > median_clicks
).astype(int)

print("Median clicks:", median_clicks)
print(feature_df["high_performance"].value_counts())

Median clicks: 0.0
high_performance
0    182310
1    182037
Name: count, dtype: int64


In [37]:
# Score WITHOUT leakage
# Predict using only the honest features:
# here we use a simple baseline: predict the majority class

majority_class = feature_df["high_performance"].mode()[0]

honest_score = (
    feature_df["high_performance"] == majority_class
).mean()

print("Score without leakage:", honest_score)

Score without leakage: 0.5003746428542022


In [38]:
# DELIBERATE LEAK
feature_df["leaked_feature"] = feature_df["high_performance"]

# Score WITH leakage
leaked_score = (
    feature_df["leaked_feature"] == feature_df["high_performance"]
).mean()

print("Score with leakage:", leaked_score)

Score with leakage: 1.0


In [39]:
# REMOVING THE LEAKED FEATURES
feature_df = feature_df.drop(columns=["leaked_feature"])

## 4. Data limits

Limitation: Only 364,347 of the 9,841,378 March page-day rows have both GSC and GA4 data available, so the five-feature analysis represents only a subset of the full content inventory.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.